# Step 2: Data Layer

Validates the schema in `src/schema.py` and the loader/normalize logic in `src/data_loader.py`.

This notebook uses small **synthetic** sample rows (not real league data) just to prove the
pipeline works end to end. Real historical season data still needs to be supplied (manual
export or API) before this can run against actual stats — see `ROADMAP.md` Step 1/2.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from data_loader import normalize_player_week, compute_fantasy_points
from schema import FULL_PPR_SCORING, LEAGUE_SETTINGS

In [2]:
# Synthetic sample data — shape matches PLAYERS_SCHEMA / WEEKLY_STATS_SCHEMA.
players_df = pd.DataFrame([
    {"player_id": 1, "name": "Sample RB", "position": "RB", "team": "AAA", "bye_week": 7, "season": 2025},
    {"player_id": 2, "name": "Sample WR", "position": "WR", "team": "BBB", "bye_week": 9, "season": 2025},
])

weekly_df = pd.DataFrame([
    {"player_id": 1, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 80, "rush_td": 1, "receptions": 3, "rec_yds": 20, "rec_td": 0,
     "special_teams_td": 0, "fumbles_lost": 0, "two_pt_conversions": 0},
    {"player_id": 2, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 0, "rush_td": 0, "receptions": 6, "rec_yds": 95, "rec_td": 1,
     "special_teams_td": 0, "fumbles_lost": 0, "two_pt_conversions": 0},
])

players_df

,player_id,name,position,team,bye_week,season
0,1,Sample RB,RB,AAA,7,2025
1,2,Sample WR,WR,BBB,9,2025


In [3]:
player_week = normalize_player_week(players_df, weekly_df)
player_week

,player_id,season,week,pass_yds,pass_td,pass_int,rush_yds,rush_td,receptions,rec_yds,rec_td,special_teams_td,fumbles_lost,two_pt_conversions,name,position,team,bye_week,fantasy_points
0,1,2025,1,0.0,0.0,0.0,80.0,1.0,3.0,20.0,0.0,0.0,0.0,0.0,Sample RB,RB,AAA,7,19.0
1,2,2025,1,0.0,0.0,0.0,0.0,0.0,6.0,95.0,1.0,0.0,0.0,0.0,Sample WR,WR,BBB,9,21.5


In [4]:
# Synthetic sample data — shape matches PLAYERS_SCHEMA / WEEKLY_STATS_SCHEMA.
players_df = pd.DataFrame([
    {"player_id": 1, "name": "Sample RB", "position": "RB", "team": "AAA", "bye_week": 7, "season": 2025},
    {"player_id": 2, "name": "Sample WR", "position": "WR", "team": "BBB", "bye_week": 9, "season": 2025},
])

weekly_df = pd.DataFrame([
    {"player_id": 1, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 80, "rush_td": 1, "receptions": 3, "rec_yds": 20, "rec_td": 0,
     "special_teams_td": 0, "fumbles_lost": 0, "two_pt_conversions": 0},
    {"player_id": 2, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 0, "rush_td": 0, "receptions": 6, "rec_yds": 95, "rec_td": 1,
     "special_teams_td": 0, "fumbles_lost": 0, "two_pt_conversions": 0},
])

players_df

,player_id,name,position,team,bye_week,season
0,1,Sample RB,RB,AAA,7,2025
1,2,Sample WR,WR,BBB,9,2025


`LEAGUE_SETTINGS` in `src/schema.py` currently holds standard Full PPR defaults (roster slots,
bench size, scoring weights) as a placeholder — still needs to be confirmed against the user's
actual league settings.

In [5]:
LEAGUE_SETTINGS

{'roster_slots': {'QB': 1,
  'RB': 2,
  'WR': 2,
  'TE': 1,
  'FLEX': 1,
  'K': 1,
  'DST': 1,
  'IR': 1},
 'bench_size': 7,
 'team_count': 12,
 'reg_season_weeks': 14,
 'playoff_team_count': 8,
 'scoring': {'pass_yds': 0.04,
  'pass_td': 4.0,
  'pass_int': -2.0,
  'rush_yds': 0.1,
  'rush_td': 6.0,
  'receptions': 1.0,
  'rec_yds': 0.1,
  'rec_td': 6.0,
  'special_teams_td': 6.0,
  'fumbles_lost': -2.0,
  'two_pt_conversions': 2.0}}

# Real Historical Data: nflverse

The synthetic demo above proves the pipeline logic works. This section loads **real** 2024
season data via `nflreadpy` (the actively-maintained nflverse data package — free, no auth
required) through `src/nflverse_loader.py`, and cross-checks our own `fantasy_points`
calculation against nflverse's independently-computed `fantasy_points_ppr` column as a
sanity check on both the scoring weights and the loader.

In [6]:
import nflverse_loader as nvl

real_players = nvl.load_players([2024])
real_weekly = nvl.load_weekly_stats([2024])
real_player_week = normalize_player_week(real_players, real_weekly)

print(f"players: {real_players.shape}, weekly_stats: {real_weekly.shape}, player_week: {real_player_week.shape}")
real_player_week.head()

players: (3215, 6), weekly_stats: (6710, 14), player_week: (6710, 19)


,player_id,season,week,pass_yds,pass_td,pass_int,rush_yds,rush_td,receptions,rec_yds,rec_td,special_teams_td,fumbles_lost,two_pt_conversions,name,position,team,bye_week,fantasy_points
0,00-0023459,2024,1,167.0,1.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Aaron Rodgers,QB,NYJ,12,8.58
1,00-0023853,2024,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matt Prater,K,ARI,11,0.00
2,00-0025565,2024,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Nick Folk,K,TEN,5,0.00
3,00-0026498,2024,1,317.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matthew Stafford,QB,LA,6,14.68
4,00-0026858,2024,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Graham Gano,K,NYG,11,0.00


In [7]:
# Cross-check our fantasy_points against nflverse's own independently-computed
# fantasy_points_ppr for the exact same games.
import nflreadpy as nfl

reference = nfl.load_player_stats([2024], summary_level="week") \
    .select(["player_id", "week", "fantasy_points_ppr"]).to_pandas()

comparison = real_player_week.merge(reference, on=["player_id", "week"], how="inner")
diff = (comparison["fantasy_points"] - comparison["fantasy_points_ppr"]).abs()

print(f"rows compared: {len(comparison)}")
print(f"max abs diff: {diff.max():.6f}")
assert (diff < 0.01).all(), "fantasy point calc disagrees with nflverse reference"
print("Full PPR scoring weights verified against real 2024 season data.")

rows compared: 6710
max abs diff: 0.000000
Full PPR scoring weights verified against real 2024 season data.
